# 01 · Data Import & Merge

Loads the 5 raw tables one at a time, inspects each one (shape, dtypes, missing values, sample rows), saves a per-table profile CSV to `src/analytics/outputs/`, then merges everything into **one loan-level DataFrame**.

Loading, cleaning and joining use the functions in `src/ingestion.py`, so this notebook and the dashboard share the same logic.

In [1]:
import sys
from pathlib import Path

# walk up from the notebook folder to the project root (where pyproject.toml lives)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import ingestion
from src.config import RAW_DIR

OUT = ROOT / 'src' / 'analytics' / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 80)

raw = ingestion.load_raw(RAW_DIR)  # dict of 5 DataFrames, IDs as str, dates parsed
list(raw)

['customers', 'loans', 'bureau', 'repayments', 'collections']

In [2]:
def profile(name, df):
    """Print an overview and save a column-level profile CSV."""
    prof = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.notna().sum(),
        'missing': df.isna().sum(),
        'missing_pct': (df.isna().mean() * 100).round(2),
        'unique': df.nunique(),
        'example': df.iloc[0],
    })
    prof.index.name = 'column'
    prof.to_csv(OUT / f'{name}_profile.csv')
    df.head(20).to_csv(OUT / f'{name}_sample.csv', index=False)
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]} columns | duplicate rows: {df.duplicated().sum()}')
    return prof

## 1. `customers.csv`

- **Grain:** 1 row per customer
- **Size:** 10,000 rows × 15 columns
- **Keys:** `customer_id` (primary key)

| Column | Description |
|---|---|
| `customer_id` | Unique customer ID |
| `age` | Age in years (21–70) |
| `gender` | Male/Female; ~6% coded `M`/`F` (DQ issue) |
| `marital_status` | Married / Single / Divorced / Widowed |
| `education_level` | Highest education |
| `employment_type` | Salaried / Self-Employed / Business Owner / Contract / Unemployed |
| `occupation_sector` | Industry sector |
| `years_employed` | Years in current job, 500 missing |
| `annual_income` | Annual income, USD |
| `region` | North / South / East / West / Central |
| `city_tier` | Tier 1 / 2 / 3 |
| `num_dependents` | Dependents, 500 missing |
| `housing_type` | Owned / Mortgaged / Rented / Living with Family |
| `years_at_residence` | Years at address, 500 missing |
| `customer_since` | Date relationship began |

In [3]:
customers = raw['customers']
profile('customers', customers)

customers: 10,000 rows x 15 columns | duplicate rows: 0


,dtype,non_null,missing,missing_pct,unique,example
column,,,,,,
customer_id,string,10000,0,0.0,10000,CUST100000
age,int64,10000,0,0.0,50,41
gender,str,10000,0,0.0,4,Female
marital_status,str,10000,0,0.0,4,Married
education_level,str,10000,0,0.0,5,Vocational
employment_type,str,10000,0,0.0,5,Business Owner
occupation_sector,str,10000,0,0.0,10,Construction
years_employed,float64,9500,500,5.0,320,2.8
annual_income,float64,10000,0,0.0,1273,87700.0


In [4]:
customers.head()

,customer_id,age,gender,marital_status,education_level,employment_type,occupation_sector,years_employed,annual_income,region,city_tier,num_dependents,housing_type,years_at_residence,customer_since
0,CUST100000,41,Female,Married,Vocational,Business Owner,Construction,2.8,87700.0,West,Tier 3,0.0,Living with Family,4.2,2021-07-25
1,CUST100001,27,Male,Single,High School,Unemployed,Healthcare,0.0,12000.0,Central,Tier 1,1.0,Living with Family,2.6,2018-11-07
2,CUST100002,46,Female,Married,Vocational,Business Owner,Finance,30.3,60100.0,Central,Tier 1,0.0,Owned,1.1,2021-08-22
3,CUST100003,48,Male,Widowed,Vocational,Unemployed,Retail,0.0,39500.0,West,Tier 3,1.0,Mortgaged,0.5,2020-06-19
4,CUST100004,21,Male,Divorced,Bachelor's,Business Owner,Finance,0.5,98500.0,Central,Tier 2,1.0,Owned,3.9,2022-06-02


In [5]:
customers.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
customer_id,10000,10000,CUST100000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,10000.0,NaN,NaN,NaN,38.192,21.0,31.0,38.0,45.0,70.0,10.4357
gender,10000,4,Male,5341,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital_status,10000,4,Married,5504,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education_level,10000,5,Bachelor's,3940,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employment_type,10000,5,Salaried,5149,NaN,NaN,NaN,NaN,NaN,NaN,NaN
occupation_sector,10000,10,Government,1047,NaN,NaN,NaN,NaN,NaN,NaN,NaN
years_employed,9500.0,NaN,NaN,NaN,5.699863,0.0,1.5,3.9,8.0,35.0,5.901365
annual_income,10000.0,NaN,NaN,NaN,79434.43,12000.0,62675.0,79400.0,96700.0,184000.0,25822.2063
region,10000,5,South,2420,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. `loan_accounts.csv`

- **Grain:** 1 row per loan account (main fact table)
- **Size:** 14,685 rows × 11 columns
- **Keys:** `account_id` (PK) → `customer_id`

| Column | Description |
|---|---|
| `account_id` | Unique loan ID |
| `customer_id` | FK → customers |
| `loan_type` | Personal / Auto / Credit Card / Mortgage / Business |
| `loan_purpose` | Stated purpose |
| `loan_amount` | Original amount |
| `interest_rate_pct` | Annual rate, 294 missing |
| `tenure_months` | Tenure, 12–360 |
| `origination_date` | Origination date |
| `disbursal_channel` | Branch / Online / Partner/DSA / Mobile App |
| `collateral_flag` | 1 = secured |
| `target_default` | **Target**: 1 = defaulted (~16%) |

In [6]:
loans = raw['loans']
profile('loans', loans)

loans: 14,685 rows x 11 columns | duplicate rows: 0


,dtype,non_null,missing,missing_pct,unique,example
column,,,,,,
account_id,string,14685,0,0.0,14685,ACC200001
customer_id,string,14685,0,0.0,10000,CUST100000
loan_type,str,14685,0,0.0,5,Mortgage
loan_purpose,str,14685,0,0.0,9,Wedding
loan_amount,float64,14685,0,0.0,14370,306310.51
interest_rate_pct,float64,14391,294,2.0,2084,6.06
tenure_months,int64,14685,0,0.0,9,360
origination_date,datetime64[us],14685,0,0.0,2186,2022-10-18 00:00:00
disbursal_channel,str,14685,0,0.0,4,Branch


In [7]:
loans.head()

,account_id,customer_id,loan_type,loan_purpose,loan_amount,interest_rate_pct,tenure_months,origination_date,disbursal_channel,collateral_flag,target_default
0,ACC200001,CUST100000,Mortgage,Wedding,306310.51,6.06,360,2022-10-18,Branch,1,1
1,ACC200002,CUST100001,Auto Loan,Home Improvement,33921.61,5.52,60,2022-12-19,Online,1,0
2,ACC200003,CUST100002,Personal Loan,Business Expansion,4469.93,10.98,48,2023-11-08,Mobile App,0,0
3,ACC200004,CUST100003,Personal Loan,Medical,26274.80,7.43,48,2020-04-06,Partner/DSA,0,0
4,ACC200005,CUST100003,Personal Loan,Vehicle Purchase,16356.86,10.63,36,2023-04-15,Online,0,0


In [8]:
loans.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
account_id,14685,14685,ACC200001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,14685,10000,CUST100026,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_type,14685,5,Personal Loan,5216,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_purpose,14685,9,Vehicle Purchase,1729,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_amount,14685.0,NaN,NaN,NaN,40934.254793,1000.0,9578.36,17334.01,31947.74,444803.02,64872.135545
interest_rate_pct,14391.0,NaN,NaN,NaN,11.874886,3.62,7.695,10.46,14.78,29.9,5.179385
tenure_months,14685.0,NaN,NaN,NaN,56.549949,12.0,24.0,36.0,48.0,360.0,71.861632
origination_date,14685,NaN,NaN,NaN,2020-12-29 06:47:08.518896,2018-01-01 00:00:00,2019-06-20 00:00:00,2021-01-04 00:00:00,2022-07-08 00:00:00,2023-12-30 00:00:00,NaN
disbursal_channel,14685,4,Mobile App,3711,NaN,NaN,NaN,NaN,NaN,NaN,NaN
collateral_flag,14685.0,NaN,NaN,NaN,0.290364,0.0,0.0,0.0,1.0,1.0,0.453946


## 3. `bureau_credit.csv`

- **Grain:** 1 row per external credit line (many per customer)
- **Size:** 22,979 rows × 10 columns
- **Keys:** `bureau_id` (PK) → `customer_id`

| Column | Description |
|---|---|
| `bureau_id` | Unique bureau record |
| `customer_id` | FK → customers |
| `credit_type` | Type of external line |
| `lender_category` | Bank / NBFC / Fintech / Cooperative |
| `sanctioned_amount` | Approved limit |
| `outstanding_amount` | Current balance |
| `account_status` | Active / Closed / Written-Off |
| `days_past_due` | DPD at last report, 689 missing |
| `date_opened` | Open date |
| `date_closed` | Close date, null if active (expected) |

In [9]:
bureau = raw['bureau']
profile('bureau', bureau)

bureau: 22,979 rows x 10 columns | duplicate rows: 0


,dtype,non_null,missing,missing_pct,unique,example
column,,,,,,
bureau_id,string,22979,0,0.00,22979,BUR300001
customer_id,string,22979,0,0.00,8992,CUST100001
credit_type,str,22979,0,0.00,6,Auto Loan
lender_category,str,22979,0,0.00,4,NBFC
sanctioned_amount,float64,22979,0,0.00,20698,7111.82
outstanding_amount,float64,22979,0,0.00,14712,1177.6
account_status,str,22979,0,0.00,3,Active
days_past_due,float64,22290,689,3.00,74,36.0
date_opened,datetime64[us],22979,0,0.00,2920,2022-07-30 00:00:00


In [10]:
bureau.head()

,bureau_id,customer_id,credit_type,lender_category,sanctioned_amount,outstanding_amount,account_status,days_past_due,date_opened,date_closed
0,BUR300001,CUST100001,Auto Loan,NBFC,7111.82,1177.60,Active,36.0,2022-07-30,NaT
1,BUR300002,CUST100001,Personal Loan,Bank,44920.95,33952.82,Active,40.0,2024-06-10,NaT
2,BUR300003,CUST100001,Retail Financing,NBFC,28440.17,24777.90,Active,42.0,2018-06-04,NaT
3,BUR300004,CUST100002,Retail Financing,Cooperative,39112.28,34998.94,Active,8.0,2018-03-15,NaT
4,BUR300005,CUST100002,Retail Financing,Fintech,34835.55,0.00,Closed,0.0,2025-12-20,2029-04-05


In [11]:
bureau.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
bureau_id,22979,22979,BUR300001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,22979,8992,CUST100180,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credit_type,22979,6,Personal Loan,3856,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lender_category,22979,4,NBFC,5926,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sanctioned_amount,22979.0,NaN,NaN,NaN,20537.158803,500.0,9942.295,19799.91,29865.305,81398.34,13608.946158
outstanding_amount,22979.0,NaN,NaN,NaN,7346.042828,0.0,0.0,3242.24,12252.44,65835.09,9484.361583
account_status,22979,3,Active,11243,NaN,NaN,NaN,NaN,NaN,NaN,NaN
days_past_due,22290.0,NaN,NaN,NaN,13.878286,0.0,0.0,8.0,25.0,78.0,15.93449
date_opened,22979,NaN,NaN,NaN,2021-12-28 10:47:50.412115,2018-01-01 00:00:00,2019-12-29 00:00:00,2021-12-29 00:00:00,2024-01-02 00:00:00,2025-12-29 00:00:00,NaN
date_closed,11736,NaN,NaN,NaN,2024-02-22 12:32:38.282208,2018-04-17 00:00:00,2022-02-26 00:00:00,2024-03-08 00:00:00,2026-02-25 00:00:00,2029-12-24 00:00:00,NaN


## 4. `repayment_history.csv`

- **Grain:** 1 row per account per month (many per account)
- **Size:** 176,220 rows × 7 columns
- **Keys:** (`account_id`, `payment_month`) → `account_id`

| Column | Description |
|---|---|
| `account_id` | FK → loan_accounts |
| `payment_month` | Month since origination (1–12) |
| `emi_due` | Installment due |
| `emi_paid` | Amount paid (0 if missed) |
| `days_past_due` | DPD that month (0–89) |
| `payment_status` | On-Time / Late / Missed |
| `outstanding_balance` | Balance after the month |

In [12]:
repayments = raw['repayments']
profile('repayments', repayments)

repayments: 176,220 rows x 7 columns | duplicate rows: 0


,dtype,non_null,missing,missing_pct,unique,example
column,,,,,,
account_id,string,176220,0,0.0,14685,ACC200001
payment_month,int64,176220,0,0.0,12,1
emi_due,float64,176220,0,0.0,14053,1848.32
emi_paid,float64,176220,0,0.0,14054,1848.32
days_past_due,int64,176220,0,0.0,89,0
payment_status,str,176220,0,0.0,3,On-Time
outstanding_balance,float64,176220,0,0.0,165159,305201.52


In [13]:
repayments.head()

,account_id,payment_month,emi_due,emi_paid,days_past_due,payment_status,outstanding_balance
0,ACC200001,1,1848.32,1848.32,0,On-Time,305201.52
1,ACC200001,2,1848.32,1848.32,14,Late,304092.53
2,ACC200001,3,1848.32,1848.32,7,Late,302983.53
3,ACC200001,4,1848.32,1848.32,0,On-Time,301874.54
4,ACC200001,5,1848.32,1848.32,0,On-Time,300765.55


In [14]:
repayments.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
account_id,176220,14685,ACC200001,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_month,176220.0,NaN,NaN,NaN,6.5,3.452062,1.0,3.75,6.5,9.25,12.0
emi_due,176220.0,NaN,NaN,NaN,880.766042,756.059408,16.91,391.27,635.17,1134.61,6956.83
emi_paid,176220.0,NaN,NaN,NaN,836.847767,757.994256,0.0,354.92,605.35,1092.03,6956.83
days_past_due,176220.0,NaN,NaN,NaN,4.47073,13.802992,0.0,0.0,0.0,0.0,89.0
payment_status,176220,3,On-Time,147369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
outstanding_balance,176220.0,NaN,NaN,NaN,37671.198995,63332.725237,314.06,7453.78,14400.61,29050.7425,442735.38


## 5. `collections_events.csv`

- **Grain:** 1 row per collections action (sparse)
- **Size:** 8,635 rows × 6 columns
- **Keys:** `collection_event_id` (PK) → `account_id`

| Column | Description |
|---|---|
| `collection_event_id` | Unique event |
| `account_id` | FK → loan_accounts |
| `event_date` | Action date |
| `action_type` | Reminder Call / SMS/Email Notice / Field Visit / Settlement Offer / Legal Notice / Written-Off Referral |
| `outcome` | Promise to Pay / Payment Received / No Response / Disputed / Escalated |
| `amount_recovered` | Recovered amount (often 0) |

In [15]:
collections = raw['collections']
profile('collections', collections)

collections: 8,635 rows x 6 columns | duplicate rows: 0


,dtype,non_null,missing,missing_pct,unique,example
column,,,,,,
collection_event_id,string,8635,0,0.0,8635,COL400001
account_id,string,8635,0,0.0,4278,ACC210780
event_date,datetime64[us],8635,0,0.0,1976,2022-07-07 00:00:00
action_type,str,8635,0,0.0,6,Field Visit
outcome,str,8635,0,0.0,5,Escalated
amount_recovered,float64,8635,0,0.0,2195,0.0


In [16]:
collections.head()

,collection_event_id,account_id,event_date,action_type,outcome,amount_recovered
0,COL400001,ACC210780,2022-07-07,Field Visit,Escalated,0.0
1,COL400002,ACC213329,2021-10-31,Reminder Call,Disputed,0.0
2,COL400003,ACC213329,2023-02-08,Legal Notice,Promise to Pay,0.0
3,COL400004,ACC213329,2021-03-21,Field Visit,Payment Received,1114.7
4,COL400005,ACC212150,2022-11-13,Reminder Call,Payment Received,0.0


In [17]:
collections.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
collection_event_id,8635,8635,COL400001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
account_id,8635,4278,ACC213329,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
event_date,8635,NaN,NaN,NaN,2021-04-11 00:33:41.169658,2018-07-20 00:00:00,2019-11-22 00:00:00,2021-04-10 00:00:00,2022-09-01 00:00:00,2023-12-30 00:00:00,NaN
action_type,8635,6,Settlement Offer,1482,NaN,NaN,NaN,NaN,NaN,NaN,NaN
outcome,8635,5,No Response,1795,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount_recovered,8635.0,NaN,NaN,NaN,665.690771,0.0,0.0,0.0,188.64,4997.74,1349.467867


## 6. Merge into one DataFrame

Output grain is **1 row per loan account**. The child tables are aggregated to their join key *before* joining, so loans are never duplicated:

```
loan_accounts (fact)
  ├─ LEFT JOIN customers             ON customer_id  (many-to-one)
  ├─ LEFT JOIN bureau aggregates     ON customer_id  (22,979 → 1 row / customer)
  ├─ LEFT JOIN repayment aggregates  ON account_id   (176,220 → 1 row / account)
  └─ LEFT JOIN collections aggregates ON account_id  (sparse, 0-filled)
```

Each join is checked with pandas `validate=`, and an assertion confirms the row count doesn't change. The data-quality fixes (gender labels, imputation + missing flags) are applied inside `build_dataset()`.

In [18]:
df, dq = ingestion.build_dataset(RAW_DIR)
print(f'Merged: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f"Unique accounts: {df['account_id'].nunique():,} | customers: {df['customer_id'].nunique():,}")
print(f"Default rate: {df['target_default'].mean():.2%}")

Merged: 14,685 rows x 68 columns
Unique accounts: 14,685 | customers: 10,000
Default rate: 16.10%


### Data-quality log

In [19]:
dq.to_csv(OUT / 'data_quality_report.csv', index=False)
dq

,table,column,issue,rows_affected,action
0,customers,*,rows loaded,10000,-
1,loans,*,rows loaded,14685,-
2,bureau,*,rows loaded,22979,-
3,repayments,*,rows loaded,176220,-
4,collections,*,rows loaded,8635,-
5,loans,customer_id,referential integrity vs customers,0,passed
6,bureau,customer_id,referential integrity vs customers,0,passed
7,repayments,account_id,referential integrity vs loans,0,passed
8,collections,account_id,referential integrity vs loans,0,passed
9,customers,gender,inconsistent labels 'M'/'F',600,mapped to Male/Female


### Merged dataset preview & checks

In [20]:
df.head()

,account_id,customer_id,loan_type,loan_purpose,loan_amount,interest_rate_pct,tenure_months,origination_date,disbursal_channel,collateral_flag,target_default,interest_rate_missing,age,gender,marital_status,education_level,employment_type,occupation_sector,years_employed,annual_income,region,city_tier,num_dependents,housing_type,years_at_residence,customer_since,years_employed_missing,num_dependents_missing,years_at_residence_missing,age_band,income_band,bureau_lines_cnt,bureau_active_cnt,bureau_writeoff_cnt,bureau_total_sanctioned,bureau_total_outstanding,bureau_max_dpd,bureau_mean_dpd,bureau_dpd_missing_cnt,bureau_lender_types,bureau_oldest_line,bureau_utilisation,bureau_has_writeoff,rep_months_observed,rep_missed_cnt,rep_late_cnt,rep_max_dpd,rep_mean_dpd,rep_total_due,rep_total_paid,rep_last_balance,rep_dpd30_cnt,rep_recent3_mean_dpd,rep_pay_ratio,rep_missed_rate,rep_dpd_trend,coll_events_cnt,coll_amount_recovered,coll_legal_or_writeoff,coll_last_event_date,has_bureau_record,loan_to_income,monthly_emi,emi_to_monthly_income,exposure,orig_year,customer_tenure_yrs,customer_loan_cnt
0,ACC200001,CUST100000,Mortgage,Wedding,306310.51,6.06,360,2022-10-18,Branch,1,1,0,41,Female,Married,Vocational,Business Owner,Construction,2.8,87700.0,West,Tier 3,0,Living with Family,4.2,2021-07-25,0,0,0,41-50,Q4,0.0,0.0,0.0,0.00,0.00,0.0,0.000000,0.0,0.0,NaT,0.000000,0.0,12,0,4,14,2.500000,22179.84,22179.84,293002.61,0,0.666667,1.000000,0.000000,-1.833333,0.0,0.0,0.0,NaT,0,3.492708,1848.32,0.252906,293002.61,2022,1.232033,1
1,ACC200002,CUST100001,Auto Loan,Home Improvement,33921.61,5.52,60,2022-12-19,Online,1,0,0,27,Male,Single,High School,Unemployed,Healthcare,0.0,12000.0,Central,Tier 1,1,Living with Family,2.6,2018-11-07,0,0,0,21-30,Q1 (lowest),3.0,3.0,0.0,80472.94,59908.32,42.0,39.333333,0.0,2.0,2018-06-04,0.744453,0.0,12,1,2,45,7.000000,7779.12,7130.86,29643.09,1,9.333333,0.916667,0.083333,2.333333,3.0,0.0,0.0,2023-06-20,1,2.826801,648.26,0.648260,29643.09,2022,4.114990,1
2,ACC200003,CUST100002,Personal Loan,Business Expansion,4469.93,10.98,48,2023-11-08,Mobile App,0,0,0,46,Female,Married,Vocational,Business Owner,Finance,30.3,60100.0,Central,Tier 1,0,Owned,1.1,2021-08-22,0,0,0,41-50,Q2,2.0,1.0,0.0,73947.83,34998.94,8.0,4.000000,0.0,2.0,2018-03-15,0.894833,0.0,12,0,1,24,2.000000,1385.76,1385.76,3638.47,0,0.000000,1.000000,0.000000,-2.000000,0.0,0.0,0.0,NaT,1,0.074375,115.48,0.023058,3638.47,2023,2.212183,1
3,ACC200004,CUST100003,Personal Loan,Medical,26274.80,7.43,48,2020-04-06,Partner/DSA,0,0,0,48,Male,Widowed,Vocational,Unemployed,Retail,0.0,39500.0,West,Tier 3,1,Mortgaged,0.5,2020-06-19,0,0,0,41-50,Q1 (lowest),3.0,1.0,1.0,50639.83,18392.44,44.0,27.000000,0.0,2.0,2018-10-08,0.261649,1.0,12,0,1,8,0.666667,7613.28,7613.28,21706.83,0,0.000000,1.000000,0.000000,-0.666667,0.0,0.0,0.0,NaT,1,0.665185,634.44,0.192741,21706.83,2020,-0.202601,2
4,ACC200005,CUST100003,Personal Loan,Vehicle Purchase,16356.86,10.63,36,2023-04-15,Online,0,0,0,48,Male,Widowed,Vocational,Unemployed,Retail,0.0,39500.0,West,Tier 3,1,Mortgaged,0.5,2020-06-19,0,0,0,41-50,Q1 (lowest),3.0,1.0,1.0,50639.83,18392.44,44.0,27.000000,0.0,2.0,2018-10-08,0.261649,1.0,12,0,1,21,1.750000,6391.68,6391.68,12521.85,0,0.000000,1.000000,0.000000,-1.750000,0.0,0.0,0.0,NaT,1,0.414098,532.64,0.161815,12521.85,2023,2.819986,2


In [21]:
assert df['account_id'].is_unique
assert len(df) == len(loans)
remaining_nulls = df.isna().sum()
remaining_nulls[remaining_nulls > 0]  # only expected date columns should remain

bureau_oldest_line       1462
coll_last_event_date    10407
dtype: int64

In [22]:
profile('merged_loan_level', df)
df.to_csv(OUT / 'merged_loan_level.csv', index=False)
print('Saved ->', OUT / 'merged_loan_level.csv')

merged_loan_level: 14,685 rows x 68 columns | duplicate rows: 0


Saved -> /home/webexpert/essex_assignment/src/analytics/outputs/merged_loan_level.csv
